In [ ]:
import os
import pandas as pd
import numpy as np
import sentencepiece as spm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import matplotlib.pyplot as plt
import random

# Reproducibility
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.1+cu130
CUDA available: True


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU Memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")

Device: cuda
GPU: Tesla T4
GPU Memory: 14.56 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
BASE_DIR = "/content/drive/MyDrive"

print("MyDrive folders:")
print(os.listdir(BASE_DIR))

DATASET_DIR = os.path.join(BASE_DIR, "Nepali_Dataset")

if os.path.exists(DATASET_DIR):
    print("\nNepali_Dataset contents:")
    print(os.listdir(DATASET_DIR))

    for folder in ["processed", "tokenizer", "checkpoints"]:
        path = os.path.join(DATASET_DIR, folder)
        print(f"\n{folder}:")
        print(os.listdir(path) if os.path.exists(path) else "Folder not found")
else:
    print("\nNepali_Dataset folder not found.")

MyDrive folders:
['Colab Notebooks', 'Nepali_Dataset']

Nepali_Dataset contents:
['small.csv', 'medium.csv', 'large.csv', 'processed', 'tokenizer', 'checkpoints']

processed:
['train.csv', 'validation.csv', 'test.csv', 'cleaned_dataset.csv']

tokenizer:
['nepali_sp.model', 'nepali_sp.vocab']

checkpoints:
[]


In [ ]:
DATASET_DIR = "/content/drive/MyDrive/Nepali_Dataset"

for filename in ["small.csv", "medium.csv", "large.csv"]:
    path = os.path.join(DATASET_DIR, filename)

    df = pd.read_csv(path)

    print(f"\n{filename}")
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())
    print(df.head(2).to_string(index=False))


small.csv
Rows: 17241
Columns: ['id', 'content', 'summary']
 id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         content                                        

In [ ]:
from sklearn.model_selection import train_test_split

# Dataset directory
DATASET_DIR = "/content/drive/MyDrive/Nepali_Dataset"
PROCESSED_DIR = os.path.join(DATASET_DIR, "processed")

os.makedirs(PROCESSED_DIR, exist_ok=True)

# Combine all three original datasets
small_df = pd.read_csv(f"{DATASET_DIR}/small.csv")
medium_df = pd.read_csv(f"{DATASET_DIR}/medium.csv")
large_df = pd.read_csv(f"{DATASET_DIR}/large.csv")

df = pd.concat(
    [small_df, medium_df, large_df],
    ignore_index=True
)

# Remove missing and empty records
df = df.dropna(subset=["content", "summary"])

df["content"] = df["content"].astype(str).str.strip()
df["summary"] = df["summary"].astype(str).str.strip()

df = df[
    (df["content"] != "") &
    (df["summary"] != "")
]

# Remove duplicate articles, keeping the first summary
df = df.drop_duplicates(
    subset=["content"],
    keep="first"
).reset_index(drop=True)

# Create 80:10:10 train-validation-test split
train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=SEED,
    shuffle=True
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=SEED,
    shuffle=True
)

# Save processed datasets
train_df.to_csv(f"{PROCESSED_DIR}/train.csv", index=False)
val_df.to_csv(f"{PROCESSED_DIR}/validation.csv", index=False)
test_df.to_csv(f"{PROCESSED_DIR}/test.csv", index=False)

# Also save the complete cleaned dataset
df.to_csv(f"{PROCESSED_DIR}/cleaned_dataset.csv", index=False)

print("Dataset preparation completed!")
print("Cleaned dataset:", len(df))
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))

Dataset preparation completed!
Cleaned dataset: 49913
Training samples: 39930
Validation samples: 4991
Testing samples: 4992


In [ ]:
for name, data in [
    ("Small", small_df),
    ("Medium", medium_df),
    ("Large", large_df)
]:
    print(f"\n{name}")
    print("Total:", len(data))
    print("Missing content:", data["content"].isna().sum())
    print("Missing summary:", data["summary"].isna().sum())
    print("Duplicate content:", data["content"].duplicated().sum())

print("\nCombined rows:", len(small_df) + len(medium_df) + len(large_df))
print("Unique articles:", df["content"].nunique())


Small
Total: 17241
Missing content: 0
Missing summary: 0
Duplicate content: 201

Medium
Total: 21885
Missing content: 0
Missing summary: 0
Duplicate content: 290

Large
Total: 11403
Missing content: 0
Missing summary: 0
Duplicate content: 124

Combined rows: 50529
Unique articles: 49913


In [ ]:
# Load revised dataset splits
DATA_DIR = "/content/drive/MyDrive/Nepali_Dataset/processed"
TOKENIZER_PATH = "/content/drive/MyDrive/Nepali_Dataset/tokenizer/nepali_sp.model"

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")
val_df = pd.read_csv(f"{DATA_DIR}/validation.csv")
test_df = pd.read_csv(f"{DATA_DIR}/test.csv")

# Load SentencePiece tokenizer
sp = spm.SentencePieceProcessor()
sp.load(TOKENIZER_PATH)

VOCAB_SIZE = sp.get_piece_size()

print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Testing samples:", len(test_df))
print("Vocabulary size:", VOCAB_SIZE)
print("Device:", device)

Training samples: 39930
Validation samples: 4991
Testing samples: 4992
Vocabulary size: 32000
Device: cuda


In [ ]:
print("Cleaned:", len(pd.read_csv(f"{DATA_DIR}/cleaned_dataset.csv")))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Cleaned: 49913
Train: 39930
Validation: 4991
Test: 4992


In [ ]:
PAD_ID = sp.pad_id()   # 0
UNK_ID = sp.unk_id()   # 1
BOS_ID = sp.bos_id()   # 2
EOS_ID = sp.eos_id()   # 3

print(f"PAD_ID: {PAD_ID}, BOS_ID: {BOS_ID}, EOS_ID: {EOS_ID}, UNK_ID: {UNK_ID}")

PAD_ID: 0, BOS_ID: 2, EOS_ID: 3, UNK_ID: 1


In [ ]:
# Fixed sequence lengths for fair comparison with baseline LSTM
MAX_CONTENT_LEN = 768
MAX_SUMMARY_LEN = 100

print("MAX_CONTENT_LEN:", MAX_CONTENT_LEN)
print("MAX_SUMMARY_LEN:", MAX_SUMMARY_LEN)

MAX_CONTENT_LEN: 768
MAX_SUMMARY_LEN: 100


In [ ]:
content_lens = train_df["content"].astype(str).apply(
    lambda x: len(sp.encode(x, out_type=int))
)

summary_lens = train_df["summary"].astype(str).apply(
    lambda x: len(sp.encode(x, out_type=int)) + 2
)

print("Source length analysis:")
print(content_lens.describe())

print("\nTarget length analysis:")
print(summary_lens.describe())

Source length analysis:
count    39930.000000
mean       353.856048
std        326.553751
min         25.000000
25%        147.000000
50%        240.000000
75%        438.000000
max       5934.000000
Name: content, dtype: float64

Target length analysis:
count    39930.000000
mean        54.167518
std         15.210120
min          9.000000
25%         45.000000
50%         54.000000
75%         64.000000
max        145.000000
Name: summary, dtype: float64


In [ ]:
print("Cleaned:", len(pd.read_csv(f"{DATA_DIR}/cleaned_dataset.csv")))
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))
print("Total:", len(train_df) + len(val_df) + len(test_df))

Cleaned: 49913
Train: 39930
Validation: 4991
Test: 4992
Total: 49913


In [ ]:
for limit in [256, 384, 512, 768, 1024]:
    coverage = (content_lens <= limit).mean() * 100
    print(f"Source length {limit}: {coverage:.2f}% retained")

print(
    f"\nTarget length 80: "
    f"{(summary_lens <= 80).mean() * 100:.2f}% retained"
)

Source length 256: 53.47% retained
Source length 384: 70.25% retained
Source length 512: 79.94% retained
Source length 768: 90.09% retained
Source length 1024: 94.83% retained

Target length 80: 95.78% retained


In [ ]:
def encode_dataframe(dataframe):
    dataframe = dataframe.copy()

    dataframe["content_ids"] = dataframe["content"].astype(str).apply(
        lambda x: sp.encode(x, out_type=int)[:MAX_CONTENT_LEN]
    )

    dataframe["summary_ids"] = dataframe["summary"].astype(str).apply(
        lambda x: (
            [BOS_ID]
            + sp.encode(x, out_type=int)[:MAX_SUMMARY_LEN - 2]
            + [EOS_ID]
        )
    )

    return dataframe


train_df = encode_dataframe(train_df)
val_df = encode_dataframe(val_df)
test_df = encode_dataframe(test_df)

print("Training sample content IDs:", train_df["content_ids"].iloc[0][:20])
print("Training sample summary IDs:", train_df["summary_ids"].iloc[0][:20])

print("\nMaximum source length:",
      train_df["content_ids"].apply(len).max())

print("Maximum summary length:",
      train_df["summary_ids"].apply(len).max())

Training sample content IDs: [338, 602, 5, 37, 4, 2064, 978, 133, 13, 76, 1198, 1930, 21, 758, 3096, 380, 21, 1408, 6, 4]
Training sample summary IDs: [2, 889, 4935, 154, 859, 79, 1449, 445, 190, 275, 861, 57, 699, 380, 7079, 8, 913, 3678, 128, 3343]

Maximum source length: 768
Maximum summary length: 100


In [ ]:
class SummarizationDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        return {
            'content': torch.tensor(row['content_ids'], dtype=torch.long),
            'summary': torch.tensor(row['summary_ids'], dtype=torch.long)
        }

In [ ]:
def collate_fn(batch):
    content_batch = [item['content'] for item in batch]
    summary_batch = [item['summary'] for item in batch]

    max_content_len = max(len(c) for c in content_batch)
    max_summary_len = max(len(s) for s in summary_batch)

    # Explicitly create tensors with dtype=torch.long
    padded_content = torch.stack([
        torch.cat([
            c,
            torch.tensor([PAD_ID] * (max_content_len - len(c)), dtype=torch.long)
        ])
        for c in content_batch
    ]).long()  # force long

    padded_summary = torch.stack([
        torch.cat([
            s,
            torch.tensor([PAD_ID] * (max_summary_len - len(s)), dtype=torch.long)
        ])
        for s in summary_batch
    ]).long()

    return {
        'content': padded_content,   # shape: (batch, max_content_len)
        'summary': padded_summary    # shape: (batch, max_summary_len)
    }

In [ ]:
BATCH_SIZE = 32# Adjust based on GPU memory

train_dataset = SummarizationDataset(train_df)
val_dataset = SummarizationDataset(val_df)
test_dataset = SummarizationDataset(test_df)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

# Check batch shape
batch = next(iter(train_loader))
print("Content batch shape:", batch['content'].shape)
print("Summary batch shape:", batch['summary'].shape)

Content batch shape: torch.Size([32, 768])
Summary batch shape: torch.Size([32, 100])


In [ ]:

from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


# Encoder
class Encoder(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_layers, dropout):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size, embedding_dim, padding_idx=PAD_ID
        )

        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_lengths):
        embedded = self.dropout(self.embedding(src))

        packed = pack_padded_sequence(
            embedded,
            src_lengths,
            batch_first=True,
            enforce_sorted=False
        )

        packed_outputs, (hidden, cell) = self.lstm(packed)

        encoder_outputs, _ = pad_packed_sequence(
            packed_outputs,
            batch_first=True,
            total_length=src.size(1)
        )

        return encoder_outputs, hidden, cell


# Memory-efficient Bahdanau Additive Attention
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.encoder_proj = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )

        self.decoder_proj = nn.Linear(
            hidden_dim, hidden_dim, bias=False
        )

        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(
        self,
        hidden,
        encoder_outputs,
        src_mask,
        encoder_features=None
    ):
        if encoder_features is None:
            encoder_features = self.encoder_proj(encoder_outputs)

        decoder_features = self.decoder_proj(hidden).unsqueeze(1)

        energy = torch.tanh(
            encoder_features + decoder_features
        )

        scores = self.v(energy).squeeze(-1)

        # FP16-safe masking
        scores = scores.masked_fill(
            ~src_mask,
            torch.finfo(scores.dtype).min
        )

        attention_weights = torch.softmax(scores, dim=1)

        return attention_weights


# Attention-based Decoder
class Decoder(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim,
        hidden_dim,
        num_layers,
        dropout,
        attention
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_ID
        )

        self.attention = attention

        self.lstm = nn.LSTM(
            embedding_dim + hidden_dim,
            hidden_dim,
            num_layers,
            batch_first=True,
            dropout=dropout
        )

        self.fc = nn.Linear(hidden_dim * 2, vocab_size)

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        trg,
        hidden,
        cell,
        encoder_outputs,
        src_mask,
        encoder_features
    ):
        embedded = self.dropout(self.embedding(trg))

        attention_weights = self.attention(
            hidden[-1],
            encoder_outputs,
            src_mask,
            encoder_features
        )

        context = torch.bmm(
            attention_weights.unsqueeze(1),
            encoder_outputs
        )

        lstm_input = torch.cat(
            (embedded, context),
            dim=2
        )

        outputs, (hidden, cell) = self.lstm(
            lstm_input,
            (hidden, cell)
        )

        predictions = self.fc(
            torch.cat((outputs, context), dim=2)
        )

        return predictions, hidden, cell, attention_weights


# Attention-based Seq2Seq
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()

        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, trg, teacher_forcing_ratio=0.5):
        batch_size = src.shape[0]
        trg_len = trg.shape[1]
        vocab_size = self.decoder.fc.out_features

        src_lengths = (src != PAD_ID).sum(dim=1).cpu()
        src_mask = src != PAD_ID

        encoder_outputs, hidden, cell = self.encoder(
            src,
            src_lengths
        )

        # Precompute encoder attention projections once
        encoder_features = self.decoder.attention.encoder_proj(
            encoder_outputs
        )

        decoder_input = trg[:, 0].unsqueeze(1)

        outputs = torch.zeros(
            batch_size,
            trg_len,
            vocab_size,
            device=self.device
        )

        for t in range(1, trg_len):
            output, hidden, cell, _ = self.decoder(
                decoder_input,
                hidden,
                cell,
                encoder_outputs,
                src_mask,
                encoder_features
            )

            outputs[:, t, :] = output.squeeze(1)

            teacher_force = random.random() < teacher_forcing_ratio
            top1 = output.argmax(2)

            decoder_input = (
                trg[:, t].unsqueeze(1)
                if teacher_force
                else top1
            )

        return outputs


In [ ]:
VOCAB_SIZE = sp.get_piece_size()
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3

encoder = Encoder(
    VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT
)

attention = Attention(HIDDEN_DIM)

decoder = Decoder(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    NUM_LAYERS,
    DROPOUT,
    attention
)

model = Seq2Seq(encoder, decoder, device).to(device)

print(model)

Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (attention): Attention(
      (encoder_proj): Linear(in_features=512, out_features=512, bias=False)
      (decoder_proj): Linear(in_features=512, out_features=512, bias=False)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (lstm): LSTM(768, 512, num_layers=2, batch_first=True, dropout=0.3)
    (fc): Linear(in_features=1024, out_features=32000, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)

criterion = nn.CrossEntropyLoss(
    ignore_index=PAD_ID
)

print("Optimizer: Adam")
print("Learning rate:", optimizer.param_groups[0]["lr"])
print("Loss function: CrossEntropyLoss")
print("Ignored padding ID:", PAD_ID)

Optimizer: Adam
Learning rate: 0.001
Loss function: CrossEntropyLoss
Ignored padding ID: 0


In [ ]:
print("Configured batch size:", BATCH_SIZE)
print("Actual train DataLoader batch size:", train_loader.batch_size)
print("Number of training batches:", len(train_loader))

batch = next(iter(train_loader))
print("Source batch shape:", batch["content"].shape)
print("Summary batch shape:", batch["summary"].shape)

print("\nGPU memory:")
print("Allocated:", round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")
print("Reserved:", round(torch.cuda.memory_reserved() / 1024**3, 2), "GB")


Configured batch size: 32
Actual train DataLoader batch size: 32
Number of training batches: 1248
Source batch shape: torch.Size([32, 751])
Summary batch shape: torch.Size([32, 100])

GPU memory:
Allocated: 0.68 GB
Reserved: 2.37 GB


In [ ]:
print("Encoder:", "Encoder" in globals())
print("Attention:", "Attention" in globals())
print("Decoder:", "Decoder" in globals())
print("Seq2Seq:", "Seq2Seq" in globals())
print("Device:", "device" in globals())
print("PAD_ID:", "PAD_ID" in globals())


Encoder: True
Attention: True
Decoder: True
Seq2Seq: True
Device: True
PAD_ID: True


In [ ]:
VOCAB_SIZE = sp.get_piece_size()
EMBEDDING_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3

encoder = Encoder(
    VOCAB_SIZE, EMBEDDING_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT
)

attention = Attention(HIDDEN_DIM)

decoder = Decoder(
    VOCAB_SIZE,
    EMBEDDING_DIM,
    HIDDEN_DIM,
    NUM_LAYERS,
    DROPOUT,
    attention
)

model = Seq2Seq(encoder, decoder, device).to(device)

print("Model initialized successfully.")
print("Model device:", next(model.parameters()).device)

Model initialized successfully.
Model device: cuda:0


In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID)

print("Optimizer and loss initialized successfully.")

Optimizer and loss initialized successfully.


In [ ]:
from tqdm.auto import tqdm

EPOCHS = 10
ACCUMULATION_STEPS = 1
best_val_loss = float("inf")

ATTENTION_CHECKPOINT_PATH = (
    "/content/drive/MyDrive/Nepali_Dataset/checkpoints/"
    "lstm_attention_best.pt"
)

os.makedirs(os.path.dirname(ATTENTION_CHECKPOINT_PATH), exist_ok=True)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=(device.type == "cuda")
)

for epoch in range(EPOCHS):

    # Training
    model.train()
    train_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS} - Training"
    )

    for batch_idx, batch in progress:

        src = batch["content"].to(device, non_blocking=True)
        trg = batch["summary"].to(device, non_blocking=True)

        group_start = (
            batch_idx // ACCUMULATION_STEPS
        ) * ACCUMULATION_STEPS

        current_accumulation = min(
            ACCUMULATION_STEPS,
            len(train_loader) - group_start
        )

        with torch.autocast(
            device_type=device.type,
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):
            output = model(
                src,
                trg,
                teacher_forcing_ratio=0.5
            )

            output = output[:, 1:].reshape(-1, VOCAB_SIZE)
            target = trg[:, 1:].reshape(-1)

            loss = criterion(output.float(), target)
            scaled_loss = loss / current_accumulation

        train_loss += loss.item()

        scaler.scale(scaled_loss).backward()

        if (
            (batch_idx + 1) % ACCUMULATION_STEPS == 0
            or batch_idx + 1 == len(train_loader)
        ):
            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{loss.item():.4f}")

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0

    with torch.no_grad():

        for batch in tqdm(
            val_loader,
            desc=f"Epoch {epoch + 1}/{EPOCHS} - Validation"
        ):
            src = batch["content"].to(device, non_blocking=True)
            trg = batch["summary"].to(device, non_blocking=True)

            with torch.autocast(
                device_type=device.type,
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):
                output = model(
                    src,
                    trg,
                    teacher_forcing_ratio=1.0
                )

                output = output[:, 1:].reshape(-1, VOCAB_SIZE)
                target = trg[:, 1:].reshape(-1)

                loss = criterion(output.float(), target)

            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader)

    print(
        f"\nEpoch {epoch + 1}/{EPOCHS} | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f}"
    )

    # Save best checkpoint
    if avg_val_loss < best_val_loss:

        best_val_loss = avg_val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scaler_state_dict": scaler.state_dict(),
            "val_loss": best_val_loss
        }, ATTENTION_CHECKPOINT_PATH)

        print("Attention model checkpoint saved.")

print("\nTraining completed.")
print("Best validation loss:", best_val_loss)
print("Checkpoint:", ATTENTION_CHECKPOINT_PATH)

Epoch 1/10 - Training:   0%|          | 0/1248 [00:00<?, ?it/s]

Epoch 1/10 - Validation:   0%|          | 0/156 [00:00<?, ?it/s]


Epoch 1/10 | Train Loss: 7.1728 | Val Loss: 6.1244
Attention model checkpoint saved.


Epoch 2/10 - Training:   0%|          | 0/1248 [00:00<?, ?it/s]

Epoch 2/10 - Validation:   0%|          | 0/156 [00:00<?, ?it/s]


Epoch 2/10 | Train Loss: 6.1856 | Val Loss: 5.3207
Attention model checkpoint saved.


... (20 outputs left out)


Epoch 9/10 | Train Loss: 4.2433 | Val Loss: 4.4351
Attention model checkpoint saved.


Epoch 10/10 - Training:   0%|          | 0/1248 [00:00<?, ?it/s]

Epoch 10/10 - Validation:   0%|          | 0/156 [00:00<?, ?it/s]


Epoch 10/10 | Train Loss: 4.1165 | Val Loss: 4.4306
Attention model checkpoint saved.

Training completed.
Best validation loss: 4.430567208008888
Checkpoint: /content/drive/MyDrive/Nepali_Dataset/checkpoints/lstm_attention_best.pt


In [ ]:
# Load the best attention LSTM checkpoint
checkpoint = torch.load(
    ATTENTION_CHECKPOINT_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

print("Best attention checkpoint loaded successfully.")
print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Model device:", next(model.parameters()).device)

Best attention checkpoint loaded successfully.
Best epoch: 10
Best validation loss: 4.430567208008888
Model device: cuda:0


In [ ]:
def generate_summary(model, source_text, max_len=MAX_SUMMARY_LEN):
    model.eval()

    src_ids = sp.encode(source_text, out_type=int)
    src_ids = src_ids[:MAX_SOURCE_LEN]

    if not src_ids:
        src_ids = [UNK_ID]

    src_tensor = torch.tensor(
        [src_ids], dtype=torch.long, device=device
    )

    src_lengths = torch.tensor([len(src_ids)], dtype=torch.long)
    src_mask = (src_tensor != PAD_ID)

    with torch.no_grad():
        # Encoder
        encoder_outputs, hidden, cell = model.encoder(
            src_tensor, src_lengths
        )

        # Precompute attention features
        encoder_features = model.decoder.attention.encoder_proj(
            encoder_outputs
        )

        # Start decoding
        decoder_input = torch.tensor(
            [[BOS_ID]], dtype=torch.long, device=device
        )

        generated_ids = []

        for _ in range(max_len):
            output, hidden, cell, _ = model.decoder(
                decoder_input,
                hidden,
                cell,
                encoder_outputs,
                src_mask,
                encoder_features
            )

            predicted_id = output.argmax(dim=-1).item()

            if predicted_id == EOS_ID:
                break

            if predicted_id not in (BOS_ID, PAD_ID):
                generated_ids.append(predicted_id)

            decoder_input = torch.tensor(
                [[predicted_id]], dtype=torch.long, device=device
            )

    return sp.decode(generated_ids)

In [ ]:
MAX_SOURCE_LEN = 768
MAX_SUMMARY_LEN = 100

In [ ]:
print("MODEL STRUCTURE:")
print(model)

print("\nSEQ2SEQ ATTRIBUTES:")
print(model._modules.keys())

print("\nDECODER ATTRIBUTES:")
print(model.decoder._modules.keys())

MODEL STRUCTURE:
Seq2Seq(
  (encoder): Encoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (lstm): LSTM(256, 512, num_layers=2, batch_first=True, dropout=0.3)
    (dropout): Dropout(p=0.3, inplace=False)
  )
  (decoder): Decoder(
    (embedding): Embedding(32000, 256, padding_idx=0)
    (attention): Attention(
      (encoder_proj): Linear(in_features=512, out_features=512, bias=False)
      (decoder_proj): Linear(in_features=512, out_features=512, bias=False)
      (v): Linear(in_features=512, out_features=1, bias=False)
    )
    (lstm): LSTM(768, 512, num_layers=2, batch_first=True, dropout=0.3)
    (fc): Linear(in_features=1024, out_features=32000, bias=True)
    (dropout): Dropout(p=0.3, inplace=False)
  )
)

SEQ2SEQ ATTRIBUTES:
dict_keys(['encoder', 'decoder'])

DECODER ATTRIBUTES:
dict_keys(['embedding', 'attention', 'lstm', 'fc', 'dropout'])


In [ ]:
sample = test_df.iloc[0]

source_text = sample["content"]
reference_summary = sample["summary"]

predicted_summary = generate_summary(model, source_text)

print("SOURCE ARTICLE:")
print(source_text)

print("\nREFERENCE SUMMARY:")
print(reference_summary)

print("\nGENERATED SUMMARY:")
print(predicted_summary)

SOURCE ARTICLE:
सर्वसाधारणको लगानीबाटै जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै घिसिङले आईपीओ नरोक्न सम्बद्ध निकायको ध्यानाकर्षण गराएका हुन् । ६ फागुन, काठमाडौं । नेपाल विद्युत प्राधिकरणका कार्यकारी निर्देशक कुलमान घिसिङले जलविद्युत् कम्पनीहरूको प्राथमिक सार्वजनिक निष्कासन आईपीओ नरोक्न आग्रह गरेका छन् । सर्वसाधारणको लगानीबाटै जलविद्युत् क्षेत्रको विकास सम्भव भएको भन्दै घिसिङले आईपीओ नरोक्न सम्बद्ध निकायको ध्यानाकर्षण गराएका हुन् । नेपाल आर्थिक पत्रकार संघले आयोजना गरेको अन्तर्क्रियामा घिसिङले आईपीओमा लगानी गर्दा जोखिम देखिए पनि नियमन गरी खोल्नुपर्ने बताए । विद्युत् प्राधिकरणले १२ हजार मेगावाटसम्मको विद्युत खरिदबिक्री सम्झौता पीपीए गरेको उनले जानकारी दिए । हाल ३ हजार ५ सय मेगावाटका आयोजना सञ्चालन भए पनि हिउँदमा १ हजार १ सय मेगावाट मात्रै उत्पादन भइरहेको उनको भनाइ छ । घिसिङले विद्युत् उत्पादन २८ हजार ५ सय मेगावाट पुर्याउने गरी प्राधिकरण अझै पीपीए खोल्ने पक्षमा रहेको बताए ।

REFERENCE SUMMARY:
जलविद्युत् आईपीओ नरोक्न कुलमानको आग्रह। सर्वसाधारणको लगानीबाटै जलविद्युत् क्षेत्रको विकास सम्

In [ ]:
import os

RESULTS_DIR = "/content/drive/MyDrive/Nepali_Dataset/results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print("Results directory ready:", RESULTS_DIR)
print("Test samples:", len(test_df))

Results directory ready: /content/drive/MyDrive/Nepali_Dataset/results
Test samples: 4992


In [ ]:
import pandas as pd
from tqdm.auto import tqdm

results = []

model.eval()

for idx, row in tqdm(
    test_df.iterrows(),
    total=len(test_df),
    desc="Generating predictions"
):
    source_text = str(row["content"])
    reference_summary = str(row["summary"])

    predicted_summary = generate_summary(model, source_text)

    results.append({
        "source": source_text,
        "reference": reference_summary,
        "prediction": predicted_summary
    })

# Save predictions
predictions_df = pd.DataFrame(results)

PREDICTION_PATH = os.path.join(
    RESULTS_DIR,
    "lstm_attention_test_predictions.csv"
)

predictions_df.to_csv(
    PREDICTION_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("\nPrediction generation completed.")
print("Total predictions:", len(predictions_df))
print("Saved to:", PREDICTION_PATH)

Generating predictions:   0%|          | 0/4992 [00:00<?, ?it/s]


Prediction generation completed.
Total predictions: 4992
Saved to: /content/drive/MyDrive/Nepali_Dataset/results/lstm_attention_test_predictions.csv


In [ ]:
!pip install -q sacrebleu rouge-score bert-score


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
import pandas as pd
import sacrebleu
from rouge_score import rouge_scorer

# Load saved predictions
predictions_df = pd.read_csv(PREDICTION_PATH).fillna("")

predictions = predictions_df["prediction"].astype(str).tolist()
references = predictions_df["reference"].astype(str).tolist()

# 1. SacreBLEU
sacrebleu_score = sacrebleu.corpus_bleu(
    predictions,
    [references]
).score

# 2. Jaccard Similarity (average per example)
def calculate_jaccard(prediction, reference):
    pred_tokens = set(prediction.split())
    ref_tokens = set(reference.split())

    union = pred_tokens | ref_tokens

    if not union:
        return 1.0

    return len(pred_tokens & ref_tokens) / len(union)

jaccard_scores = [
    calculate_jaccard(pred, ref)
    for pred, ref in zip(predictions, references)
]

jaccard_score = sum(jaccard_scores) / len(jaccard_scores)

# 3. ROUGE
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False
)

rouge_results = [
    scorer.score(ref, pred)
    for pred, ref in zip(references, predictions)
]

rouge1 = sum(r["rouge1"].fmeasure for r in rouge_results) / len(rouge_results)
rouge2 = sum(r["rouge2"].fmeasure for r in rouge_results) / len(rouge_results)
rougeL = sum(r["rougeL"].fmeasure for r in rouge_results) / len(rouge_results)

# Display results
print("ATTENTION LSTM — TEST RESULTS")
print("=" * 40)
print(f"Test samples : {len(predictions)}")
print(f"SacreBLEU    : {sacrebleu_score:.4f}")
print(f"Jaccard      : {jaccard_score:.4f}")
print(f"ROUGE-1      : {rouge1:.4f}")
print(f"ROUGE-2      : {rouge2:.4f}")
print(f"ROUGE-L      : {rougeL:.4f}")

ATTENTION LSTM — TEST RESULTS
Test samples : 4992
SacreBLEU    : 7.1501
Jaccard      : 0.1870
ROUGE-1      : 0.0024
ROUGE-2      : 0.0024
ROUGE-L      : 0.0024


In [ ]:
import os
import pandas as pd

PREDICTION_PATH = (
    "/content/drive/MyDrive/Nepali_Dataset/results/"
    "lstm_attention_test_predictions.csv"
)

print("File exists:", os.path.exists(PREDICTION_PATH))

if os.path.exists(PREDICTION_PATH):
    df = pd.read_csv(PREDICTION_PATH)
    print("Rows:", len(df))
    print("Columns:", df.columns.tolist())

File exists: False


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

RESULTS_DIR = "/content/drive/MyDrive/Nepali_Dataset/results"

print("Results folder exists:", os.path.exists(RESULTS_DIR))

if os.path.exists(RESULTS_DIR):
    print("Files:", os.listdir(RESULTS_DIR))

Results folder exists: True
Files: ['lstm_attention_test_predictions.csv']


In [ ]:
import pandas as pd

PREDICTION_PATH = os.path.join(
    RESULTS_DIR,
    "lstm_attention_test_predictions.csv"
)

predictions_df = pd.read_csv(PREDICTION_PATH).fillna("")

predictions = predictions_df["prediction"].astype(str).tolist()
references = predictions_df["reference"].astype(str).tolist()

print("Predictions loaded:", len(predictions))
print("References loaded:", len(references))
print("Columns:", predictions_df.columns.tolist())

Predictions loaded: 4992
References loaded: 4992
Columns: ['source', 'reference', 'prediction']


In [ ]:
!pip install -q rouge-score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
from rouge_score import rouge_scorer

class NepaliTokenizer:
    def tokenize(self, text):
        return text.strip().split()

scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=False,
    tokenizer=NepaliTokenizer()
)

rouge_results = [
    scorer.score(ref, pred)
    for pred, ref in zip(references, predictions)
]

rouge1 = sum(r["rouge1"].fmeasure for r in rouge_results) / len(rouge_results)
rouge2 = sum(r["rouge2"].fmeasure for r in rouge_results) / len(rouge_results)
rougeL = sum(r["rougeL"].fmeasure for r in rouge_results) / len(rouge_results)

print("NEPALI ROUGE RESULTS")
print("=" * 35)
print(f"ROUGE-1 : {rouge1:.4f}")
print(f"ROUGE-2 : {rouge2:.4f}")
print(f"ROUGE-L : {rougeL:.4f}")

NEPALI ROUGE RESULTS
ROUGE-1 : 0.2703
ROUGE-2 : 0.1025
ROUGE-L : 0.2294
